# 🔬 Lab Module 2: LoanBot v0.2 — Full 5-Layer Harness

**Mục tiêu:** Nâng cấp LoanBot từ v0.1 (minimal dispatcher) lên v0.2 với **5-layer production harness** đầy đủ.

## Những gì ta sẽ xây:
| Layer | Components | LoanBot v0.2 |
|-------|-----------|---------------|
| 1 - Tool Orchestration | ToolRegistry, RetryOrchestrator, CircuitBreaker, BudgetTracker | ✅ |
| 2 - Verification | OutputValidator | ✅ |
| 3 - Context & Memory | MemoryManager (compaction), StateStore | ✅ |
| 4 - Guardrails | PermissionResolver, InputValidator, HITLGateway | ✅ |
| 5 - Observability | AuditLogger (JSON structured logs) | ✅ |

## Prerequisite
```bash
pip install anthropic
export ANTHROPIC_API_KEY='your-key-here'
```

In [ ]:
# Cell 1: Imports và cấu hình
import anthropic
import json
import time
import random
import uuid
import copy
from datetime import datetime
from typing import Optional
from dataclasses import dataclass, field, asdict
from enum import Enum

MODEL = "claude-haiku-4-5-20251001"
MAX_TOKENS = 4096
client = anthropic.Anthropic()
print(f"✅ Anthropic SDK ready. Model: {MODEL}")

---
## 🔵 LAYER 1: Tool Orchestration
### Mục tiêu: ToolRegistry + RetryOrchestrator + CircuitBreaker + BudgetTracker

In [ ]:
# Cell 2: CircuitBreaker
class CBState(Enum):
    CLOSED    = "CLOSED"     # hoạt động bình thường
    OPEN      = "OPEN"       # ngắt - fail fast
    HALF_OPEN = "HALF_OPEN"  # thử nghiệm recovery

class CircuitBreaker:
    """Circuit Breaker pattern để ngăn thundering herd."""
    def __init__(self, failure_threshold=3, recovery_timeout=10):
        self.failure_threshold = failure_threshold
        self.recovery_timeout  = recovery_timeout
        self.state             = CBState.CLOSED
        self.failure_count     = 0
        self.last_failure_time = None

    def can_call(self) -> bool:
        if self.state == CBState.CLOSED:
            return True
        if self.state == CBState.OPEN:
            elapsed = time.time() - self.last_failure_time
            if elapsed >= self.recovery_timeout:
                self.state = CBState.HALF_OPEN  # thử recovery
                return True
            return False  # fail fast
        return True  # HALF_OPEN: thử 1 request

    def record_success(self):
        self.state         = CBState.CLOSED
        self.failure_count = 0

    def record_failure(self):
        self.failure_count    += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.failure_threshold or self.state == CBState.HALF_OPEN:
            self.state = CBState.OPEN

    def status(self):
        return f"{self.state.value} (failures: {self.failure_count})"

# Demo
cb = CircuitBreaker(failure_threshold=3, recovery_timeout=5)
for i in range(4):
    print(f"Can call: {cb.can_call()} | State: {cb.status()}")
    cb.record_failure()
print(f"After 4 failures: {cb.status()}")
print(f"Fail fast check: {cb.can_call()}")

In [ ]:
# Cell 3: ToolRegistry + RetryOrchestrator
class ToolRegistry:
    """Kho trung tâm đăng ký tools với metadata đầy đủ."""
    def __init__(self):
        self._tools = {}

    def register(self, name, fn, schema, permission="auto",
                 max_retries=3, base_delay=0.5):
        self._tools[name] = {
            "fn":          fn,
            "schema":      schema,
            "permission":  permission,
            "max_retries": max_retries,
            "base_delay":  base_delay,
            "call_count":  0,
            "error_count": 0,
            "available":   True,
            "cb":          CircuitBreaker(failure_threshold=3)
        }

    def get_schemas(self):
        return [t["schema"] for t in self._tools.values() if t["available"]]

    def get(self, name):
        return self._tools.get(name)

    def stats(self):
        return {n: {"calls": t["call_count"], "errors": t["error_count"],
                    "cb": t["cb"].status()}
                for n, t in self._tools.items()}


def retry_call(fn, kwargs, max_retries=3, base_delay=0.5, tool_name=""):
    """Exponential backoff + jitter retry."""
    for attempt in range(max_retries):
        try:
            return fn(**kwargs)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            wait = base_delay * (2 ** attempt) + random.uniform(0, 0.2)
            print(f"  ⚠️  {tool_name} attempt {attempt+1} failed: {e}. Retry in {wait:.1f}s...")
            time.sleep(wait)

print("✅ ToolRegistry + RetryOrchestrator ready")

In [ ]:
# Cell 4: BudgetTracker
class BudgetTracker:
    """Theo dõi tool calls và chi phí. Dừng agent khi vượt ngưỡng."""
    def __init__(self, max_tool_calls=20, max_iterations=10, max_cost_usd=0.5):
        self.max_tool_calls  = max_tool_calls
        self.max_iterations  = max_iterations
        self.max_cost_usd    = max_cost_usd
        self.tool_calls      = 0
        self.iterations      = 0
        self.input_tokens    = 0
        self.output_tokens   = 0

    def record_iteration(self):
        self.iterations += 1

    def record_tool_call(self):
        self.tool_calls += 1

    def record_tokens(self, inp, out):
        self.input_tokens  += inp
        self.output_tokens += out

    @property
    def estimated_cost_usd(self):
        # Haiku pricing: $0.25/M input, $1.25/M output
        return (self.input_tokens * 0.25 + self.output_tokens * 1.25) / 1_000_000

    def check_limits(self):
        """Returns (ok: bool, reason: str)"""
        if self.iterations >= self.max_iterations:
            return False, f"Max iterations reached ({self.max_iterations})"
        if self.tool_calls >= self.max_tool_calls:
            return False, f"Max tool calls reached ({self.max_tool_calls})"
        if self.estimated_cost_usd >= self.max_cost_usd:
            return False, f"Cost limit reached (${self.estimated_cost_usd:.4f})"
        return True, "ok"

    def summary(self):
        return (f"Iterations: {self.iterations}/{self.max_iterations} | "
                f"Tool calls: {self.tool_calls} | "
                f"Tokens: {self.input_tokens}in/{self.output_tokens}out | "
                f"Cost: ${self.estimated_cost_usd:.4f}")

print("✅ BudgetTracker ready")

---
## 🟡 LAYER 4: Guardrails & Safety
### PermissionResolver + InputValidator + HITLGateway

In [ ]:
# Cell 5: PermissionResolver + InputValidator
class PermissionLevel(Enum):
    AUTO          = "auto"          # thực thi tự động
    VALIDATE      = "validate"      # validate trước
    REQUIRE_HITL  = "require_hitl"  # cần human approval
    BLOCKED       = "blocked"       # từ chối hoàn toàn


# Input validation rules cho LoanBot
VALIDATION_RULES = {
    "check_credit_score": {
        "customer_id": lambda v: isinstance(v, str) and v.startswith("FC-"),
    },
    "calculate_dti": {
        "monthly_income":  lambda v: isinstance(v, (int, float)) and 1 <= v <= 500,
        "requested_loan":  lambda v: isinstance(v, (int, float)) and 10 <= v <= 5000,
        "term_months":     lambda v: v in [12, 24, 36, 48, 60, 84, 120],
    },
    "generate_report": {
        "decision": lambda v: v in ["APPROVED", "REJECTED", "NEEDS_HUMAN_REVIEW"],
    }
}

# HITL thresholds cho LoanBot (FinTech Corp policy)
HITL_THRESHOLDS = {
    "generate_report": {
        "loan_amount": 500,  # triệu VNĐ — vay >500tr cần duyệt
    }
}


def validate_input(tool_name, inputs):
    """Returns (is_valid, error_msg)"""
    for field, check in VALIDATION_RULES.get(tool_name, {}).items():
        if field in inputs and not check(inputs[field]):
            return False, f"Invalid value: {field}={inputs[field]}"
    return True, ""


def check_hitl_required(tool_name, inputs):
    """Returns True if action requires human approval."""
    thresholds = HITL_THRESHOLDS.get(tool_name, {})
    for field, limit in thresholds.items():
        if inputs.get(field, 0) > limit:
            return True, f"{field}={inputs[field]} > threshold {limit}"
    return False, ""


# Test
ok, msg = validate_input("calculate_dti", {"monthly_income": 28.5, "requested_loan": 200, "term_months": 18})
print(f"Validation test (term=18): valid={ok}, msg='{msg}'")

ok, msg = validate_input("calculate_dti", {"monthly_income": 28.5, "requested_loan": 200, "term_months": 24})
print(f"Validation test (term=24): valid={ok}")

hitl, reason = check_hitl_required("generate_report", {"loan_amount": 800, "decision": "APPROVED"})
print(f"HITL check (800tr): required={hitl}, reason='{reason}'")

---
## 🟢 LAYER 3: Context & Memory
### MemoryManager (compaction) + StateStore

In [ ]:
# Cell 6: MemoryManager
class MemoryManager:
    """
    Quản lý working memory với compaction khi context gần đầy.
    Trong lab này: ước tính tokens bằng len(str)//4.
    """
    COMPACTION_THRESHOLD = 0.75

    def __init__(self, max_tokens=20_000, client_ref=None):
        self.max_tokens  = max_tokens
        self.client      = client_ref
        self.compactions = 0

    def estimate_tokens(self, messages):
        return sum(len(str(m)) // 4 for m in messages)

    def should_compact(self, messages):
        used = self.estimate_tokens(messages)
        ratio = used / self.max_tokens
        return ratio >= self.COMPACTION_THRESHOLD, ratio

    def compact(self, messages):
        """Nén lịch sử cũ thành summary, giữ system prompt + 4 messages gần nhất."""
        if len(messages) <= 5:
            return messages  # quá ít để compact

        system_msg  = messages[0]   # giữ nguyên system prompt
        recent_msgs = messages[-4:] # giữ 4 messages gần nhất
        old_msgs    = messages[1:-4]

        # Tạo summary từ old messages
        history_text = "\n".join([
            f"{m.get('role','?')}: {str(m.get('content',''))[:200]}"
            for m in old_msgs
        ])

        if self.client:
            resp = self.client.messages.create(
                model=MODEL, max_tokens=300,
                messages=[{"role": "user", "content":
                    f"Tóm tắt ngắn gọn (dưới 150 từ) các key facts từ lịch sử sau, "
                    f"tập trung vào số liệu và kết quả:\n{history_text}"}]
            )
            summary_text = resp.content[0].text
        else:
            # Fallback khi không có client (test)
            summary_text = f"[SUMMARY of {len(old_msgs)} messages: history condensed]"

        self.compactions += 1
        before = len(messages)
        compacted = [
            system_msg,
            {"role": "assistant", "content": f"[COMPACTED HISTORY - Compaction #{self.compactions}]\n{summary_text}"},
            *recent_msgs
        ]
        print(f"  🗜️  Memory compacted: {before} → {len(compacted)} messages (compaction #{self.compactions})")
        return compacted

    def maybe_compact(self, messages):
        should, ratio = self.should_compact(messages)
        if should:
            print(f"  📊 Context at {ratio:.0%} — triggering compaction")
            return self.compact(messages)
        return messages


print("✅ MemoryManager ready")

# Demo compaction threshold
mm = MemoryManager(max_tokens=1000)
fake_msgs = [{"role": "user", "content": "x" * 100}] * 8  # 8 messages × 100 chars
should, ratio = mm.should_compact(fake_msgs)
print(f"Demo: 8 large messages → ratio={ratio:.0%}, should_compact={should}")

---
## 🟠 LAYER 5: Observability & Audit
### Structured JSON Audit Logger + Trace ID

In [ ]:
# Cell 7: AuditLogger
@dataclass
class AuditEvent:
    event_id:        str
    trace_id:        str
    timestamp:       str
    event_type:      str
    agent_id:        str = "loanbot-v0.2"
    tool_name:       Optional[str] = None
    input_data:      Optional[dict] = None
    output_data:     Optional[dict] = None
    latency_ms:      Optional[float] = None
    status:          str = "success"
    permission_level:str = "auto"
    retry_count:     int = 0
    message:         Optional[str] = None

    def to_json(self):
        return json.dumps(asdict(self), ensure_ascii=False, indent=2)


class AuditLogger:
    """Structured JSON audit logger với trace ID."""
    def __init__(self, trace_id=None):
        self.trace_id = trace_id or f"trace-{uuid.uuid4().hex[:12]}"
        self.events   = []

    def log(self, event_type, **kwargs):
        event = AuditEvent(
            event_id  = f"evt-{uuid.uuid4().hex[:8]}",
            trace_id  = self.trace_id,
            timestamp = datetime.utcnow().isoformat() + "Z",
            event_type= event_type,
            **kwargs
        )
        self.events.append(event)
        return event

    def log_tool_call(self, tool_name, input_data, output_data, latency_ms,
                      status="success", permission_level="auto", retry_count=0):
        return self.log("tool_call", tool_name=tool_name, input_data=input_data,
                        output_data=output_data, latency_ms=round(latency_ms, 2),
                        status=status, permission_level=permission_level,
                        retry_count=retry_count)

    def log_hitl(self, tool_name, reason, decision):
        return self.log("hitl_gate", tool_name=tool_name,
                        message=f"HITL triggered: {reason} → {decision}")

    def log_session(self, event_type, **kwargs):
        return self.log(event_type, **kwargs)

    def get_audit_trail(self):
        return [asdict(e) for e in self.events]

    def print_summary(self):
        print(f"\n📋 Audit Trail [{self.trace_id}] — {len(self.events)} events:")
        for e in self.events:
            status_icon = "✅" if e.status == "success" else "❌" if e.status == "error" else "ℹ️"
            tool_info = f" [{e.tool_name}]" if e.tool_name else ""
            latency   = f" ({e.latency_ms:.0f}ms)" if e.latency_ms else ""
            msg       = f" — {e.message}" if e.message else ""
            print(f"  {status_icon} {e.event_type}{tool_info}{latency}{msg}")


# Demo logger
logger = AuditLogger(trace_id="trace-demo-001")
logger.log_session("session_start", message="LoanBot v0.2 session started")
logger.log_tool_call("check_credit_score",
    input_data={"customer_id": "FC-2024-001"},
    output_data={"score": 720, "risk_level": "thấp"},
    latency_ms=245)
logger.print_summary()
print("\n📄 Sample event JSON:")
print(logger.events[1].to_json())

---
## 🛠️ Tool Schemas & Mock Implementations
(Kế thừa từ Module 1, giữ nguyên để v0.2 backward-compatible)

In [ ]:
# Cell 8: Tool Schemas (Anthropic format)
LOANBOT_TOOL_SCHEMAS = [
    {
        "name": "check_credit_score",
        "description": "Kiểm tra điểm tín dụng của khách hàng từ hệ thống CIC Việt Nam.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string", "description": "Mã khách hàng (FC-XXXX-XXX)"},
                "include_history": {"type": "boolean", "description": "Có lấy lịch sử không"}
            },
            "required": ["customer_id"]
        }
    },
    {
        "name": "verify_income",
        "description": "Xác minh thu nhập với BHXH và ngân hàng.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id":      {"type": "string"},
                "declared_income":  {"type": "number", "description": "Thu nhập khai báo (triệu VNĐ/tháng)"}
            },
            "required": ["customer_id", "declared_income"]
        }
    },
    {
        "name": "check_blacklist",
        "description": "Kiểm tra danh sách đen nợ xấu.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "string"},
                "national_id": {"type": "string", "description": "Số CMND/CCCD"}
            },
            "required": ["customer_id"]
        }
    },
    {
        "name": "calculate_dti",
        "description": "Tính Debt-to-Income ratio. DTI ≤ 43% là tiêu chuẩn FinTech Corp.",
        "input_schema": {
            "type": "object",
            "properties": {
                "monthly_income":      {"type": "number"},
                "existing_monthly_debt":{"type": "number"},
                "requested_loan":      {"type": "number"},
                "term_months":         {"type": "integer"},
                "annual_rate":         {"type": "number"}
            },
            "required": ["monthly_income", "existing_monthly_debt", "requested_loan", "term_months"]
        }
    },
    {
        "name": "generate_report",
        "description": "Tạo báo cáo thẩm định chính thức.",
        "input_schema": {
            "type": "object",
            "properties": {
                "customer_id":       {"type": "string"},
                "loan_amount":       {"type": "number"},
                "decision":          {"type": "string", "enum": ["APPROVED", "REJECTED", "NEEDS_HUMAN_REVIEW"]},
                "findings_summary":  {"type": "string"},
                "justification":     {"type": "string"}
            },
            "required": ["customer_id", "loan_amount", "decision", "findings_summary"]
        }
    }
]

print(f"✅ {len(LOANBOT_TOOL_SCHEMAS)} tool schemas defined")

In [ ]:
# Cell 9: Mock tool implementations
MOCK_DB = {
    "FC-2024-001": {"name": "Nguyễn Văn An",  "credit": 720, "income": 28.5, "debt": 3.0, "blacklisted": False, "national_id": "001001234567"},
    "FC-2024-002": {"name": "Trần Thị Bình",  "credit": 580, "income": 15.0, "debt": 4.5, "blacklisted": False, "national_id": "002001234567"},
    "FC-2024-003": {"name": "Lê Văn Cường",   "credit": 650, "income": 20.0, "debt": 2.0, "blacklisted": True,  "national_id": "003001234567"},
}

# Inject failures để test retry/circuit breaker
_cic_fail_counter = {"count": 0}

def mock_check_credit_score(customer_id, include_history=True):
    # Simulate 30% failure rate để test retry
    _cic_fail_counter["count"] += 1
    if _cic_fail_counter["count"] == 2:  # fail lần 2 để test retry
        raise TimeoutError("CIC system timeout (simulated)")
    c = MOCK_DB.get(customer_id, {})
    if not c: return {"error": f"Customer {customer_id} not found"}
    return {"customer_id": customer_id, "name": c["name"],
            "score": c["credit"], "risk_level": "thấp" if c["credit"] >= 700 else "trung bình" if c["credit"] >= 600 else "cao",
            "meets_minimum": c["credit"] >= 650}

def mock_verify_income(customer_id, declared_income):
    c = MOCK_DB.get(customer_id, {})
    if not c: return {"error": f"Customer {customer_id} not found"}
    verified = c["income"]
    confidence = 0.95 if abs(verified - declared_income) / declared_income < 0.1 else 0.70
    return {"customer_id": customer_id, "declared_income": declared_income,
            "verified_income": verified, "confidence": confidence,
            "verified": confidence >= 0.80}

def mock_check_blacklist(customer_id, national_id=None):
    c = MOCK_DB.get(customer_id, {})
    if not c: return {"error": f"Customer {customer_id} not found"}
    return {"customer_id": customer_id, "is_blacklisted": c["blacklisted"],
            "reason": "Nợ xấu nhóm 5 tại VPBank 2023" if c["blacklisted"] else None}

def mock_calculate_dti(monthly_income, existing_monthly_debt, requested_loan,
                       term_months=60, annual_rate=12.0):
    monthly_rate = annual_rate / 100 / 12
    new_payment  = requested_loan * monthly_rate * (1 + monthly_rate)**term_months / ((1 + monthly_rate)**term_months - 1)
    total_debt   = existing_monthly_debt + new_payment
    dti          = (total_debt / monthly_income) * 100
    return {"monthly_income": monthly_income, "existing_debt": existing_monthly_debt,
            "new_payment": round(new_payment, 3), "total_monthly_debt": round(total_debt, 3),
            "dti_ratio_pct": round(dti, 2), "passes_dti_check": dti <= 43.0}

def mock_generate_report(customer_id, loan_amount, decision, findings_summary, justification=""):
    report_id = f"RPT-{customer_id}-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    return {"report_id": report_id, "customer_id": customer_id, "loan_amount": loan_amount,
            "decision": decision, "timestamp": datetime.now().isoformat(),
            "findings": findings_summary, "report_url": f"https://fintech.vn/reports/{report_id}"}

TOOL_FNS = {
    "check_credit_score": mock_check_credit_score,
    "verify_income":      mock_verify_income,
    "check_blacklist":    mock_check_blacklist,
    "calculate_dti":      mock_calculate_dti,
    "generate_report":    mock_generate_report,
}

print("✅ Mock tool implementations ready (check_credit has 1 simulated failure for retry demo)")

---
## 🏗️ Tổng hợp: LoanBotHarnessV2
### Kết hợp tất cả 5 layers thành một harness hoàn chỉnh

In [ ]:
# Cell 10: LoanBotHarnessV2 — Full 5-Layer Harness
class LoanBotHarnessV2:
    """
    Production harness đầy đủ 5 tầng cho LoanBot v0.2.
    Tích hợp: ToolRegistry, RetryOrchestrator, CircuitBreaker,
              BudgetTracker, MemoryManager, AuditLogger,
              PermissionResolver, InputValidator, HITLGateway.
    """

    SYSTEM_PROMPT = """Bạn là LoanBot v0.2 của FinTech Corp — AI agent thẩm định hồ sơ vay.

QUYTRÌNH BẮT BUỘC (thực hiện theo đúng thứ tự):
1. check_credit_score → phải ≥ 650
2. verify_income → xác minh thu nhập
3. check_blacklist → phải không trong danh sách đen
4. calculate_dti → DTI ≤ 43% là đạt
5. generate_report → kết luận APPROVED / REJECTED / NEEDS_HUMAN_REVIEW

TUYỆT ĐỐI KHÔNG:
- Tự tính toán hay đoán kết quả tool
- Bỏ qua bất kỳ bước nào
- Phê duyệt hồ sơ có blacklist = True
- Phê duyệt khoản vay > 500 triệu (dùng NEEDS_HUMAN_REVIEW)"""

    def __init__(self, hitl_mode="auto"):
        """
        hitl_mode: 'auto' = tự động approve HITL cho demo
                   'manual' = dừng và hỏi người dùng
        """
        self.registry  = ToolRegistry()
        self.budget    = BudgetTracker(max_iterations=10, max_tool_calls=15)
        self.memory    = MemoryManager(max_tokens=30_000, client_ref=client)
        self.hitl_mode = hitl_mode
        self._register_tools()

    def _register_tools(self):
        # Permission levels cho từng tool (FinTech Corp policy)
        perms = {
            "check_credit_score": "auto",
            "verify_income":      "auto",
            "check_blacklist":    "auto",
            "calculate_dti":      "validate",
            "generate_report":    "validate",  # HITL check done dynamically
        }
        for schema in LOANBOT_TOOL_SCHEMAS:
            name = schema["name"]
            self.registry.register(
                name       = name,
                fn         = TOOL_FNS[name],
                schema     = schema,
                permission = perms.get(name, "auto"),
                max_retries= 3 if name == "check_credit_score" else 2,
                base_delay = 0.1  # nhỏ để lab chạy nhanh
            )

    def dispatch_tool(self, tool_name, tool_input, logger):
        """Layer 1+2+4: Orchestration + Validation + Guardrails."""
        tool_meta = self.registry.get(tool_name)
        if not tool_meta:
            return {"error": f"Tool '{tool_name}' not found in registry"}

        # Layer 4a: Permission check
        if tool_meta["permission"] == "blocked":
            logger.log("permission_blocked", tool_name=tool_name,
                       message=f"BLOCKED: {tool_name} is not allowed")
            return {"error": f"Permission BLOCKED for {tool_name}"}

        # Layer 4b: Input validation
        if tool_meta["permission"] in ["validate", "require_hitl"]:
            valid, err = validate_input(tool_name, tool_input)
            if not valid:
                logger.log("validation_error", tool_name=tool_name, message=err, status="error")
                return {"error": f"Validation failed: {err}"}

        # Layer 4c: HITL check
        needs_hitl, hitl_reason = check_hitl_required(tool_name, tool_input)
        if needs_hitl:
            print(f"  👤 HITL TRIGGERED: {hitl_reason}")
            if self.hitl_mode == "manual":
                decision = input(f"  ⚠️  Approve this action? (yes/no): ").strip().lower()
                hitl_decision = "APPROVED" if decision == "yes" else "REJECTED"
            else:
                hitl_decision = "APPROVED"  # auto-approve for demo
                print(f"  🤖 HITL auto-approved (demo mode)")
            logger.log_hitl(tool_name, hitl_reason, hitl_decision)
            if hitl_decision == "REJECTED":
                return {"error": "HITL_REJECTED", "reason": hitl_reason}

        # Layer 1: Circuit Breaker check
        cb = tool_meta["cb"]
        if not cb.can_call():
            logger.log("circuit_breaker_open", tool_name=tool_name,
                       message="Circuit OPEN — fail fast", status="error")
            return {"error": f"Circuit breaker OPEN for {tool_name}"}

        # Layer 1: Retry + Execution
        start = time.time()
        retry_count = 0
        tool_meta["call_count"] += 1
        self.budget.record_tool_call()

        try:
            result = retry_call(
                fn         = tool_meta["fn"],
                kwargs     = tool_input,
                max_retries= tool_meta["max_retries"],
                base_delay = tool_meta["base_delay"],
                tool_name  = tool_name
            )
            latency = (time.time() - start) * 1000
            cb.record_success()

            logger.log_tool_call(
                tool_name        = tool_name,
                input_data       = tool_input,
                output_data      = result,
                latency_ms       = latency,
                permission_level = tool_meta["permission"],
                retry_count      = retry_count
            )
            return result

        except Exception as e:
            cb.record_failure()
            tool_meta["error_count"] += 1
            latency = (time.time() - start) * 1000
            logger.log_tool_call(tool_name=tool_name, input_data=tool_input,
                                 output_data=None, latency_ms=latency, status="error",
                                 message=str(e))
            return {"error": f"{type(e).__name__}: {e}"}

    def run(self, loan_request):
        """Main agent loop với đầy đủ 5-layer harness."""
        trace_id  = f"trace-{uuid.uuid4().hex[:12]}"
        logger    = AuditLogger(trace_id=trace_id)
        messages  = [{"role": "user", "content": loan_request}]

        print(f"\n{'='*60}")
        print(f"🚀 LoanBot v0.2 | Trace: {trace_id}")
        print(f"📋 Request: {loan_request[:80]}..." if len(loan_request) > 80 else f"📋 {loan_request}")
        print('='*60)

        logger.log_session("session_start", message=loan_request[:200])
        final_response = None

        while True:
            # Budget check
            ok, limit_reason = self.budget.check_limits()
            if not ok:
                print(f"\n🛑 BUDGET LIMIT: {limit_reason}")
                logger.log_session("budget_exceeded", message=limit_reason, status="error")
                break

            self.budget.record_iteration()

            # Memory compaction check (Layer 3)
            messages = self.memory.maybe_compact(
                [{"role": "system", "content": self.SYSTEM_PROMPT}] + messages
            )
            # Extract system prompt back
            if messages[0].get("role") == "system":
                system_content = messages[0]["content"]
                messages = messages[1:]
            else:
                system_content = self.SYSTEM_PROMPT

            # Call Claude
            response = client.messages.create(
                model      = MODEL,
                max_tokens = MAX_TOKENS,
                system     = system_content,
                tools      = self.registry.get_schemas(),
                messages   = messages
            )
            self.budget.record_tokens(response.usage.input_tokens, response.usage.output_tokens)

            # Process response
            if response.stop_reason == "end_turn":
                final_response = next(
                    (b.text for b in response.content if hasattr(b, "text")), ""
                )
                print(f"\n✅ LoanBot final response:\n{final_response}")
                messages.append({"role": "assistant", "content": response.content})
                logger.log_session("session_end", message=final_response[:300])
                break

            if response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                tool_results = []

                for block in response.content:
                    if block.type == "tool_use":
                        print(f"  🔧 Calling: {block.name}({json.dumps(block.input, ensure_ascii=False)[:60]}...)")
                        result = self.dispatch_tool(block.name, block.input, logger)
                        print(f"     → {json.dumps(result, ensure_ascii=False)[:80]}")
                        tool_results.append({
                            "type":        "tool_result",
                            "tool_use_id": block.id,
                            "content":     json.dumps(result, ensure_ascii=False)
                        })

                messages.append({"role": "user", "content": tool_results})
            else:
                print(f"\n⚠️ Unexpected stop_reason: {response.stop_reason}")
                break

        # Final audit summary
        print(f"\n📊 Budget: {self.budget.summary()}")
        logger.print_summary()
        print(f"\n🗂️  Tool stats: {json.dumps(self.registry.stats(), indent=2)}")
        return final_response, logger.get_audit_trail()


print("✅ LoanBotHarnessV2 class defined — all 5 layers integrated")

---
## 🧪 Test Cases
### Test 1: Hồ sơ tốt — FC-2024-001 (Nguyễn Văn An, vay 200 triệu)

In [ ]:
# Cell 11: Test Case 1 — Good profile
# Reset failure counter
_cic_fail_counter["count"] = 0

harness = LoanBotHarnessV2(hitl_mode="auto")

loan_request_1 = """
Hồ sơ vay vốn:
- Khách hàng: Nguyễn Văn An
- Mã KH: FC-2024-001
- CMND: 001001234567
- Thu nhập khai báo: 28.5 triệu VNĐ/tháng
- Nợ hiện tại: 3.0 triệu VNĐ/tháng
- Số tiền vay: 200 triệu VNĐ
- Kỳ hạn: 60 tháng
- Lãi suất: 12% năm

Vui lòng thẩm định hồ sơ và đưa ra quyết định.
"""

response_1, audit_trail_1 = harness.run(loan_request_1)

print(f"\n🔍 Ghi chú: Lần gọi check_credit_score thứ 2 sẽ fail và được retry tự động (test exponential backoff)")

### Test 2: Hồ sơ có blacklist — FC-2024-003

In [ ]:
# Cell 12: Test Case 2 — Blacklisted customer
_cic_fail_counter["count"] = 0
harness2 = LoanBotHarnessV2(hitl_mode="auto")

loan_request_2 = """
Hồ sơ vay vốn:
- Khách hàng: Lê Văn Cường
- Mã KH: FC-2024-003
- CMND: 003001234567
- Thu nhập khai báo: 20.0 triệu VNĐ/tháng
- Nợ hiện tại: 2.0 triệu VNĐ/tháng
- Số tiền vay: 150 triệu VNĐ
- Kỳ hạn: 60 tháng
- Lãi suất: 12% năm

Vui lòng thẩm định hồ sơ.
"""

response_2, audit_trail_2 = harness2.run(loan_request_2)
print("\n💡 LoanBot phải từ chối vì check_blacklist = True")

### Test 3: Hồ sơ vay lớn — Trigger HITL Gateway (>500 triệu)

In [ ]:
# Cell 13: Test Case 3 — Large loan triggering HITL
_cic_fail_counter["count"] = 0
harness3 = LoanBotHarnessV2(hitl_mode="auto")

loan_request_3 = """
Hồ sơ vay vốn — KHOẢN VAY LỚN:
- Khách hàng: Nguyễn Văn An
- Mã KH: FC-2024-001
- CMND: 001001234567
- Thu nhập khai báo: 28.5 triệu VNĐ/tháng
- Nợ hiện tại: 3.0 triệu VNĐ/tháng
- Số tiền vay: 800 triệu VNĐ  ← KHOẢN VAY LỚN
- Kỳ hạn: 120 tháng (10 năm)
- Lãi suất: 10% năm

Vui lòng thẩm định hồ sơ. Lưu ý: đây là khoản vay lớn cần xem xét kỹ.
"""

response_3, audit_trail_3 = harness3.run(loan_request_3)
print("\n💡 HITL Gateway phải được trigger khi LoanBot gọi generate_report với loan_amount=800 triệu")

---
## 📋 Phân Tích Audit Trail
Minh họa sức mạnh của Observability Layer

In [ ]:
# Cell 14: Audit trail analysis
print("="*60)
print("📊 PHÂN TÍCH AUDIT TRAIL — Test Case 1 (FC-2024-001)")
print("="*60)

if audit_trail_1:
    tool_events = [e for e in audit_trail_1 if e["event_type"] == "tool_call"]
    
    print(f"\n📈 Tổng events: {len(audit_trail_1)}")
    print(f"🔧 Tool calls: {len(tool_events)}")
    
    if tool_events:
        total_latency = sum(e["latency_ms"] or 0 for e in tool_events)
        print(f"⏱️  Total tool latency: {total_latency:.0f}ms")
        print(f"\n📋 Tool call breakdown:")
        for e in tool_events:
            status = "✅" if e["status"] == "success" else "❌"
            retry  = f" (retry:{e['retry_count']})" if e.get("retry_count", 0) > 0 else ""
            print(f"   {status} {e['tool_name']:25} {e['latency_ms']:6.0f}ms {e['permission_level']:10}{retry}")
    
    # Simulate compliance query: "Tại sao FC-2024-001 được duyệt?"
    print("\n🔍 Compliance Query: 'Kết quả thẩm định FC-2024-001?'")
    trace_id = audit_trail_1[0]["trace_id"]
    print(f"   trace_id: {trace_id}")
    for e in audit_trail_1:
        if e["event_type"] == "tool_call" and e["output_data"]:
            out = e["output_data"]
            if "score" in out:
                print(f"   Credit score: {out.get('score')} ({out.get('risk_level')})")
            if "dti_ratio_pct" in out:
                print(f"   DTI ratio: {out.get('dti_ratio_pct')}% (passes: {out.get('passes_dti_check')})")
            if "decision" in out:
                print(f"   Decision: {out.get('decision')} (report: {out.get('report_id')})")
else:
    print("⚠️ No audit trail available (API call may have failed)")

---
## 📝 Bài Tập

### Bài Tập 1: Thêm Output Validator (Layer 2)
Hiện tại LoanBot v0.2 chưa validate kết quả trả về từ tools (Layer 2 — Verification Loop). Implement function `validate_tool_output(tool_name, output)` và tích hợp vào `dispatch_tool`.

**Yêu cầu:**
- `check_credit_score` → score phải trong khoảng 300–850
- `calculate_dti` → dti_ratio_pct phải > 0
- `generate_report` → decision phải trong enum hợp lệ

**Gợi ý:** Tương tự `VALIDATION_RULES` nhưng cho output thay vì input.

In [ ]:
# Bài Tập 1: Implement Output Validator

OUTPUT_VALIDATION_RULES = {
    "check_credit_score": {
        # TODO: Thêm rule: score phải trong 300-850
    },
    "calculate_dti": {
        # TODO: Thêm rule: dti_ratio_pct > 0
    },
    "generate_report": {
        # TODO: Thêm rule: decision phải trong ["APPROVED", "REJECTED", "NEEDS_HUMAN_REVIEW"]
    }
}

def validate_tool_output(tool_name, output):
    """TODO: Implement output validation. Returns (is_valid, error_msg)"""
    # Gợi ý: tương tự validate_input() nhưng cho output
    pass

# Test:
# ok, msg = validate_tool_output("check_credit_score", {"score": 999})
# print(f"Test invalid score 999: {ok}, {msg}")

### Bài Tập 2: Thêm Cost Alert vào BudgetTracker

Implement một method `alert_if_expensive(threshold_usd=0.01)` trong `BudgetTracker` — in cảnh báo khi estimated_cost vượt ngưỡng. Sau đó gọi method này sau mỗi lần `record_tokens()`.

**Gợi ý:** Dùng `self.estimated_cost_usd` property đã có.

In [ ]:
# Bài Tập 2: Cost Alert

# TODO: Subclass BudgetTracker và thêm method alert_if_expensive()
class BudgetTrackerWithAlerts(BudgetTracker):
    
    def alert_if_expensive(self, threshold_usd=0.01):
        """TODO: Print warning nếu cost vượt threshold"""
        pass

# Test:
# bt = BudgetTrackerWithAlerts()
# bt.record_tokens(50000, 5000)  # Sẽ trigger alert
# bt.alert_if_expensive(threshold_usd=0.01)

### Bài Tập 3 (Nâng Cao): Manual HITL Mode

Chạy lại Test Case 3 (khoản vay 800 triệu) với `hitl_mode="manual"` — nghĩa là HarnessV2 sẽ hỏi bạn có approve không. Thử cả 2 trường hợp:
1. Gõ `yes` → LoanBot tiếp tục và tạo báo cáo NEEDS_HUMAN_REVIEW
2. Gõ `no` → LoanBot nhận HITL_REJECTED và báo cáo từ chối

**Mục đích:** Hiểu luồng HITL trong production — agent dừng và chờ, không tự quyết định.

In [ ]:
# Bài Tập 3: Manual HITL
# Uncomment để chạy (yêu cầu input từ keyboard)

# _cic_fail_counter["count"] = 0
# harness_manual = LoanBotHarnessV2(hitl_mode="manual")
# response, trail = harness_manual.run(loan_request_3)
# print("\n💡 Thử gõ 'yes' hoặc 'no' khi được hỏi để thấy HITL gateway hoạt động")

---
## 🎓 Tổng Kết Module 2

### So sánh LoanBot v0.1 vs v0.2:

| Tính năng | v0.1 (Module 1) | v0.2 (Module 2) |
|-----------|----------------|----------------|
| Tool dispatch | ✅ dispatch_tool() | ✅ ToolRegistry.dispatch_tool() |
| Retry khi lỗi | ❌ Crash ngay | ✅ Exponential backoff (3 lần) |
| Circuit breaker | ❌ | ✅ CLOSED/OPEN/HALF-OPEN |
| Budget control | ❌ | ✅ Max iterations + cost limit |
| Memory management | ❌ | ✅ Compaction khi 75% đầy |
| Permission control | ❌ | ✅ AUTO/VALIDATE/HITL/BLOCKED |
| HITL gateway | ❌ | ✅ Trigger khi >500 triệu |
| Input validation | ❌ | ✅ Validate trước dispatch |
| Audit logging | ❌ | ✅ Structured JSON + trace_id |

### Modules tiếp theo:
- **Module 3:** Evaluate LoanBot v0.2 với 12-metric framework
- **Module 4:** Deploy LoanBot lên production với Docker + CI/CD
- **Module 5:** Governance, EU AI Act compliance cho LoanBot
- **Module 6:** Multi-agent: LoanBot v2.0 với Supervisor + 4 Specialist Agents

In [ ]:
# Cell 15: Final checklist
print("✅ Module 2 Lab Checklist")
print("="*50)
items = [
    "CircuitBreaker với 3 trạng thái CLOSED/OPEN/HALF-OPEN",
    "ToolRegistry với metadata + health status",
    "RetryOrchestrator với exponential backoff + jitter",
    "BudgetTracker giới hạn iterations/calls/cost",
    "MemoryManager với compaction threshold 75%",
    "AuditLogger structured JSON với trace_id",
    "PermissionResolver 4 tiers",
    "InputValidator với VALIDATION_RULES",
    "HITLGateway trigger >500 triệu",
    "LoanBotHarnessV2 tích hợp tất cả 5 layers",
]
for item in items:
    print(f"   ☑ {item}")

print("\n🏁 Sẵn sàng cho Module 3: Đánh giá & Testing!")